<a href="https://colab.research.google.com/github/lerkalarionova2018-rgb/nlp-homeworks/blob/main/%D0%9B%D0%B0%D1%80%D0%B8%D0%BE%D0%BD%D0%BE%D0%B2%D0%B0_%22rnn_homework_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: генерация текста с помощью RNN (собственный корпус)

## Задача

1. Собрать свой небольшой текстовый корпус (не менее 1000 предложений) с помощью библиотеки `requests` (парсинг новостного сайта, блога, форума и т.д.).
2. Обучить рекуррентную нейросеть (RNN/LSTM) на собранных данных для генерации текста (по образцу из приложенного ноутбука `Copy_of_rnn.ipynb`).
3. После обучения вывести на экран 2–3 сгенерированных предложения.
4. *Дополнительно (на 5 баллов, но не обязательно):* посчитать метрику перплексии (perplexity) на валидационной выборке.
5. *Для себя (не оценивается):* обучить модель с использованием GPU.

## Критерии оценки

- **3 балла** — корпус собран (≥1000 предложений), модель обучена, сгенерировано хотя бы 1 предложение.
- **4 балла** — всё из п.3 + код с комментариями, объясняющими ключевые этапы.
- **5 баллов** — всё из п.4 + дополнительно посчитана метрика перплексии.

## Важно

- Качество сгенерированного текста не оценивается.
- Выберите **свой уникальный сайт** для парсинга и укажите его в отчёте.
- Не используйте готовые датасеты из интернета.


## 1. Установка и импорт библиотек

In [1]:
!pip install beautifulsoup4 requests lxml -q

import requests
from bs4 import BeautifulSoup
import time
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow версия:", tf.__version__)
print("GPU доступна:", tf.config.list_physical_devices('GPU'))

TensorFlow версия: 2.20.0
GPU доступна: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Парсинг текстового корпуса

**Ваш уникальный сайт:** https://lenta.ru

**Обоснование выбора:** Я выбрала сайт Lenta.ru, чтобы генерировать заголовки и краткие новостные заметки на русском языке.

In [7]:
def scrape_corpus(base_url, num_pages=5):
    """
    Парсинг текстового корпуса с Lenta.ru.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    texts = []

    for page in range(1, num_pages + 1):
        url = f"{base_url}?page={page}"
        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')

            # Селекторы для Lenta.ru - только новости
            for tag in soup.find_all('h3'):
                text = tag.get_text(strip=True)
                if text and len(text) > 10 and len(text) < 200:
                    texts.append(text)

            for tag in soup.find_all('p'):
                text = tag.get_text(strip=True)
                if text and 30 < len(text) < 500:
                    if not any(word in text for word in ['Cookie', 'cookie', 'реклама', 'Реклама', 'Войти', 'Ok']):
                        texts.append(text)

            print(f"Страница {page}: собрано {len(texts)} текстов")
            time.sleep(1)

        except Exception as e:
            print(f"Ошибка на странице {page}: {e}")

    return texts


base_url = "https://lenta.ru"
corpus = scrape_corpus(base_url, num_pages=7)

print(f"\nВсего собрано текстов: {len(corpus)}")
print("Примеры:", corpus[:5])

Страница 1: собрано 157 текстов
Страница 2: собрано 314 текстов
Страница 3: собрано 471 текстов
Страница 4: собрано 628 текстов
Страница 5: собрано 785 текстов
Страница 6: собрано 942 текстов
Страница 7: собрано 1099 текстов

Всего собрано текстов: 1099
Примеры: ['В Генштабе сообщили о контроле ВС РФ подавляющей части территории Красного Лимана', 'Стало известно о задержке поездов из-за перекрытия Крымского моста', 'Огромная белая акула унесла жизнь туриста в Западной Австралии', 'Hongqi раскрыл характеристики «Подсолнуха» для россиян', 'Песков начинал учить турецкий язык «в виде наказания»']


## 3. Подготовка данных для RNN

Токенизация, создание последовательностей, паддинг.

In [9]:
# Создаём токенизатор
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

# Преобразуем тексты в последовательности чисел
sequences = tokenizer.texts_to_sequences(corpus)

# Создаём входные и выходные данные для causal language modeling
X, y = [], []
for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

# Паддинг
X = pad_sequences(X)

# One-hot encoding для y
vocab_size = len(tokenizer.word_index) + 1
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

print(f"Размер входных данных X: {X.shape}")
print(f"Размер выходных данных y: {y.shape}")
print(f"Размер словаря: {vocab_size}")

Размер входных данных X: (8946, 20)
Размер выходных данных y: (8946, 897)
Размер словаря: 897


## 4. Создание и обучение модели RNN (LSTM)

In [15]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=100, input_length=X.shape[1]))
model.add(LSTM(150, return_sequences=False))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# Обучение (можно увеличить эпохи при хороших результатах)
history = model.fit(X, y, epochs=10, batch_size=32, validation_split=0.2)

print("\nОбучение завершено!")

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
224/224 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.0454 - loss: 6.3874 - val_accuracy: 0.0480 - val_loss: 5.8929
Epoch 2/10
224/224 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.0602 - loss: 5.4103 - val_accuracy: 0.0832 - val_loss: 4.8143
Epoch 3/10
224/224 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.1620 - loss: 4.2966 - val_accuracy: 0.2877 - val_loss: 3.6927
Epoch 4/10
224/224 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3875 - loss: 3.1959 - val_accuracy: 0.5402 - val_loss: 2.6909
Epoch 5/10
224/224 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.6298 - loss: 2.2509 - val_accuracy: 0.7721 - val_loss: 1.8537
Epoch 6/10
224/224 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8132 - loss: 1.5078 - val_accuracy: 0.8788 - val_loss: 1.2387
Epoch 7/10
224/224 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9089 - loss: 0.9890 - val_accuracy: 0.9257 - val_loss: 0.8211
Epoch 8/10
224/224 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9396 - loss: 0.6573 - val_accuracy: 

## 5. Генерация текста

Функция генерации и вывод 2–3 предложений.

In [19]:
def generate_text(seed_text, next_words=10, max_sequence_len=50):

    generated = seed_text

    for _ in range(next_words):
        # Преобразуем seed в последовательность чисел
        token_list = tokenizer.texts_to_sequences([generated])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        # Предсказываем следующее слово
        predicted_probs = model.predict(token_list, verbose=0)[0]
        predicted_index = np.argmax(predicted_probs)

        # Находим слово по индексу
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        # Добавляем слово к сгенерированному тексту
        generated += " " + output_word

    return generated

print("ГЕНЕРАЦИЯ ТЕКСТА")

# Генерируем 3 предложения с разными начальными фразами
seeds = ["В России", "Президент заявил", "Сегодня в мире"]

for i, seed in enumerate(seeds, 1):
    print(f"Пример {i}")
    print(f"Начальная фраза: '{seed}'")
    generated = generate_text(seed, next_words=8, max_sequence_len=50)
    print(f"Сгенерированный текст: '{generated}'")

ГЕНЕРАЦИЯ ТЕКСТА
Пример 1
Начальная фраза: 'В России'
Сгенерированный текст: 'В России испытали ракетный комплекс «сармат» «сармат» оказался на грани'
Пример 2
Начальная фраза: 'Президент заявил'
Сгенерированный текст: 'Президент заявил в законе тюрик стал главной целью для киллеров'
Пример 3
Начальная фраза: 'Сегодня в мире'
Сгенерированный текст: 'Сегодня в мире сша предупредили украину о возможностях россии россии с'


## 6. (Дополнительно, на 5 баллов) Расчёт перплексии

Перплексия = exp(loss). Чем ниже, тем лучше модель предсказывает последовательность.

In [22]:
# Оценка модели на валидационных данных
loss, accuracy = model.evaluate(X, y, verbose=0)
perplexity = np.exp(loss)

print("РАСЧЁТ ПЕРПЛЕКСИИ")
print(f"Потери (loss): {loss:.4f}")
print(f"Точность (accuracy): {accuracy:.4f}")
print(f"ПЕРПЛЕКСИЯ: {perplexity:.4f}")
print("Пояснение:")
print("- Перплексия показывает, насколько модель 'удивлена' тестовыми данными")
print("- Чем ниже значение, тем лучше модель предсказывает следующее слово")
print(f"- Наша модель: в среднем выбирает из {perplexity:.2f} возможных слов")

РАСЧЁТ ПЕРПЛЕКСИИ
Потери (loss): 0.2797
Точность (accuracy): 0.9577
ПЕРПЛЕКСИЯ: 1.3227
Пояснение:
- Перплексия показывает, насколько модель 'удивлена' тестовыми данными
- Чем ниже значение, тем лучше модель предсказывает следующее слово
- Наша модель: в среднем выбирает из 1.32 возможных слов


## 7. GPU (не оценивается)

Убедитесь, что обучение запускалось на GPU:
```python
print("GPU доступна:", tf.config.list_physical_devices('GPU'))
```

В Google Colab: Среда выполнения - Изменить тип среды выполнения - Выберите T4 GPU.

Документация: https://www.tensorflow.org/guide/gpu

In [23]:
print("Устройства GPU:", tf.config.list_physical_devices('GPU'))

Устройства GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
